# 11d. Causal Inference: Compute Debiased Treatment

**Role of this notebook.** Compute the debiased treatment variables for the causal test transitions using the masked toy recommender trained in notebook 11c.

This notebook loads the 11c model and head weights, scores candidate videos for each test session, simulates a toy recommendation category distribution, updates beliefs under real and toy category counts, and saves:

$$
D_\mu = \mathrm{imp\_mean}^{real}_{after} - \mathrm{imp\_mean}^{toy}_{after},
$$

$$
D_\sigma = \mathrm{imp\_std}^{real}_{after} - \mathrm{imp\_std}^{toy}_{after}.
$$

This notebook does **not** train or calibrate any model and does **not** run the final DML regression.

Default execution is a smoke test. Set `RUN_FULL_TREATMENT = True` after running full 11c to write production treatment outputs.


In [1]:
from pathlib import Path
import json
import random

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 200)

ROOT = Path('/Users/haozhangao/Desktop/RecSys Research')
DATA_DIR = ROOT / 'KuaiRec 2.0' / 'data' / 'processed'
OUT_DIR = ROOT / 'python_code_new' / 'outputs'

PREDICTIONS_PATH = OUT_DIR / 'causal_delta_tau_predictions.parquet'
TRANSITIONS_PATH = OUT_DIR / 'causal_tau_choice_transitions.parquet'
STRUCTURAL_INTERACTIONS_PATH = OUT_DIR / 'structural_interactions.parquet'
STRUCTURAL_THETA_PHI_PATH = OUT_DIR / 'structural_estimates_theta_phi.npz'
STRUCTURAL_ARRAYS_PATH = OUT_DIR / 'structural_estimation_arrays.npz'
PROCESSED_DATA_PATH = DATA_DIR / 'processed_data.parquet'

MODEL_PATH = OUT_DIR / 'causal_masked_toy_mlp.pt'
PREPROCESS_PATH = OUT_DIR / 'causal_masked_toy_mlp_preprocess.joblib'
FEATURES_PATH = OUT_DIR / 'causal_masked_toy_mlp_features.json'
HEAD_WEIGHTS_PATH = OUT_DIR / 'causal_masked_head_weights.json'
MASKED_SESSIONS_PATH = OUT_DIR / 'causal_masked_test_sessions.parquet'

SMOKE_MODEL_PATH = OUT_DIR / 'causal_masked_toy_mlp_smoke.pt'
SMOKE_PREPROCESS_PATH = OUT_DIR / 'causal_masked_toy_mlp_preprocess_smoke.joblib'
SMOKE_FEATURES_PATH = OUT_DIR / 'causal_masked_toy_mlp_features_smoke.json'
SMOKE_HEAD_WEIGHTS_PATH = OUT_DIR / 'causal_masked_head_weights_smoke.json'
SMOKE_MASKED_SESSIONS_PATH = OUT_DIR / 'causal_masked_test_sessions_smoke.parquet'

TREATMENT_OUTPUT_PATH = OUT_DIR / 'causal_debiased_treatment_transitions.parquet'
METRICS_OUTPUT_PATH = OUT_DIR / 'causal_debiased_treatment_metrics.json'
SMOKE_TREATMENT_OUTPUT_PATH = OUT_DIR / 'causal_debiased_treatment_transitions_smoke.parquet'
SMOKE_METRICS_OUTPUT_PATH = OUT_DIR / 'causal_debiased_treatment_metrics_smoke.json'

RUN_FULL_TREATMENT = False
SMOKE_TEST_N_SESSIONS = 5
EXCLUDE_PREVIOUSLY_SEEN = True
SCORE_BATCH_SIZE = 16_384
PRIOR_ALPHA = 1.0
RANDOM_SEED = 20260701

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

if RUN_FULL_TREATMENT:
    model_path = MODEL_PATH
    preprocess_path = PREPROCESS_PATH
    features_path = FEATURES_PATH
    head_weights_path = HEAD_WEIGHTS_PATH
    masked_sessions_path = MASKED_SESSIONS_PATH
    treatment_output_path = TREATMENT_OUTPUT_PATH
    metrics_output_path = METRICS_OUTPUT_PATH
else:
    model_path = SMOKE_MODEL_PATH
    preprocess_path = SMOKE_PREPROCESS_PATH
    features_path = SMOKE_FEATURES_PATH
    head_weights_path = SMOKE_HEAD_WEIGHTS_PATH
    masked_sessions_path = SMOKE_MASKED_SESSIONS_PATH
    treatment_output_path = SMOKE_TREATMENT_OUTPUT_PATH
    metrics_output_path = SMOKE_METRICS_OUTPUT_PATH

for p in [PREDICTIONS_PATH, TRANSITIONS_PATH, STRUCTURAL_INTERACTIONS_PATH, STRUCTURAL_THETA_PHI_PATH, STRUCTURAL_ARRAYS_PATH, PROCESSED_DATA_PATH, model_path, preprocess_path, features_path, head_weights_path, masked_sessions_path]:
    if not p.exists():
        raise FileNotFoundError(p)

print('device:', DEVICE)
print('mode:', 'FULL' if RUN_FULL_TREATMENT else 'SMOKE')


device: mps
mode: SMOKE


## 1. Load Test Transitions and 11c Artifacts

The outcome residual `delta_tau_resid` is loaded from notebook 11b and is not modified here. The model, preprocessing, feature list, masked sessions, and head weights are loaded from notebook 11c.


In [2]:
pred = pd.read_parquet(PREDICTIONS_PATH)
pred['burst_id'] = pred['burst_id'].astype(int)
transitions_all = pd.read_parquet(TRANSITIONS_PATH)
struct_interactions = pd.read_parquet(STRUCTURAL_INTERACTIONS_PATH)
masked_sessions = pd.read_parquet(masked_sessions_path)
prep = joblib.load(preprocess_path)
features = json.loads(features_path.read_text())
head_weight_payload = json.loads(head_weights_path.read_text())
head_weights = head_weight_payload['weights']
checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)

test_pred = pred[pred['split'].eq('test')].copy()
join_cols = ['user_id', 'burst_id', 'session', 'next_session']
extra_cols = ['sess_idx_t', 'n_interactions_t'] + [c for c in transitions_all.columns if c.startswith('B_k_') and c.endswith('_t')]
test_full = test_pred.merge(transitions_all[join_cols + extra_cols], on=join_cols, how='left', validate='one_to_one')
if RUN_FULL_TREATMENT:
    test_transitions = test_full.copy()
else:
    test_transitions = test_full.head(SMOKE_TEST_N_SESSIONS).copy()

continuous_cols = features['continuous_cols']
binary_cols = features['binary_cols']
categorical_cols = features['categorical_cols']
label_cols = features['label_cols']

theta_phi_npz = np.load(STRUCTURAL_THETA_PHI_PATH, allow_pickle=True)
arrays_npz = np.load(STRUCTURAL_ARRAYS_PATH, allow_pickle=True)
cat_ids = theta_phi_npz['cat_ids'].astype(np.int64)
reference_cat_idx = int(theta_phi_npz['reference_cat_idx'][0])
cat_to_idx = {int(cid): idx for idx, cid in enumerate(cat_ids)}
mu_by_user = theta_phi_npz['theta_b_cat'][None, :] + theta_phi_npz['theta_U'] @ theta_phi_npz['theta_V'].T
mu_by_user = mu_by_user - mu_by_user[:, [reference_cat_idx]]
sigmas_by_user = arrays_npz['sigmas_by_user'].astype(np.float64)

print('all test transitions:', f'{len(test_full):,}')
print('transitions used this run:', f'{len(test_transitions):,}')
print('head weights:', head_weights)


all test transitions: 4,069
transitions used this run: 5
head weights: {'w_complete': -7.213309461851266, 'w_long': 51.70591308659878, 'w_rewatch': -5.756700022456218, 'w_neg': 35.62549380660338}


## 2. Rebuild Model Class and Preprocessing Transform

The code below mirrors 11c architecture and preprocessing. No training happens in this notebook.


In [3]:
def transform_frame(frame, prep):
    cont_cols = prep['continuous_cols']
    bin_cols = prep['binary_cols']
    cat_cols = prep['categorical_cols']
    cont = frame[cont_cols].apply(pd.to_numeric, errors='coerce')
    for col in cont_cols:
        cont[col] = cont[col].fillna(prep['cont_median'][col])
        cont[col] = (cont[col] - prep['cont_mean'][col]) / prep['cont_std'][col]
    bins = frame[bin_cols].apply(pd.to_numeric, errors='coerce')
    for col in bin_cols:
        bins[col] = bins[col].fillna(prep['bin_median'][col])
    x_cat = np.zeros((len(frame), len(cat_cols)), dtype=np.int64)
    for j, col in enumerate(cat_cols):
        vals = frame[col].astype('object').where(frame[col].notna(), '__MISSING__').astype(str)
        x_cat[:, j] = vals.map(prep['cat_maps'][col]).fillna(0).to_numpy(dtype=np.int64)
    return cont.to_numpy(dtype=np.float32), bins.to_numpy(dtype=np.float32), x_cat

class EdgeMLP(nn.Module):
    def __init__(self, n_cont, n_bin, cat_cardinalities, hidden_sizes, dropout=0.1, emb_dim_cap=32):
        super().__init__()
        self.embeddings = nn.ModuleList()
        emb_dims = []
        for card in cat_cardinalities:
            dim = min(emb_dim_cap, max(4, int(round(card ** 0.25 * 8))))
            self.embeddings.append(nn.Embedding(int(card), dim))
            emb_dims.append(dim)
        input_dim = int(n_cont) + int(n_bin) + int(sum(emb_dims))
        layers = []
        last = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers.append(nn.Linear(last, 4))
        self.net = nn.Sequential(*layers)

    def forward(self, x_cont, x_bin, x_cat):
        parts = [x_cont, x_bin]
        parts.extend([emb(x_cat[:, j]) for j, emb in enumerate(self.embeddings)])
        return self.net(torch.cat(parts, dim=1))

model = EdgeMLP(len(continuous_cols), len(binary_cols), checkpoint['cat_cardinalities'], checkpoint['hidden_sizes'], dropout=checkpoint['dropout']).to(DEVICE)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

@torch.no_grad()
def predict_probs(frame):
    x_cont, x_bin, x_cat = transform_frame(frame, prep)
    tensors = [torch.from_numpy(x_cont), torch.from_numpy(x_bin), torch.from_numpy(x_cat)]
    loader = DataLoader(TensorDataset(*tensors), batch_size=SCORE_BATCH_SIZE, shuffle=False)
    out = []
    for batch in loader:
        x_cont_b, x_bin_b, x_cat_b = [b.to(DEVICE) for b in batch]
        out.append(torch.sigmoid(model(x_cont_b, x_bin_b, x_cat_b)).cpu().numpy())
    return np.vstack(out)


## 3. Reconstruct Alpha, Real Counts, and Candidate State

The pre-session Dirichlet counts are reconstructed from the same vanilla belief rule as notebook 05:

$$
\alpha_{before,t,k}=1+\sum_{s<t} count_{i,s,k}.
$$


In [4]:
test_session_keys = test_transitions[['user_id', 'burst_id', 'session']].drop_duplicates().copy()
test_key_set = set(map(tuple, test_session_keys.astype(int).to_numpy()))
test_users = sorted(test_session_keys['user_id'].unique().tolist())
state_cols = sorted(set(['user_id', 'video_id', 'burst_id', 'session', 'time', 'i_cat_level1_id', 'y_complete'] + continuous_cols + binary_cols + categorical_cols))
proc_test = pd.read_parquet(PROCESSED_DATA_PATH, columns=state_cols)
proc_test['burst_id'] = proc_test['burst_id'].astype(int)
proc_test['session'] = proc_test['session'].astype(int)
proc_test = proc_test[proc_test['user_id'].isin(test_users)].copy()

session_counts_all = proc_test.groupby(['user_id', 'burst_id', 'session', 'i_cat_level1_id'], observed=True).size().rename('n').reset_index()
alpha_before = {}
seen_before = {}
cat_ema_before = {}
EMA_ALPHA = 0.1
for uid, user_rows in tqdm(proc_test.sort_values(['user_id', 'time', 'video_id']).groupby('user_id', sort=False), desc='Rebuild alpha progress', unit='user'):
    alpha = np.full(len(cat_ids), PRIOR_ALPHA, dtype=np.float64)
    seen = set()
    ema_by_cat = {}
    sessions_order = user_rows[['burst_id', 'session']].drop_duplicates().sort_values(['burst_id', 'session'])
    counts_lookup = {}
    for r in session_counts_all[session_counts_all['user_id'].eq(uid)].itertuples(index=False):
        key = (int(r.burst_id), int(r.session))
        counts_lookup.setdefault(key, np.zeros(len(cat_ids), dtype=np.float64))
        counts_lookup[key][cat_to_idx[int(r.i_cat_level1_id)]] += float(r.n)
    session_rows_lookup = {key: g for key, g in user_rows.groupby(['burst_id', 'session'], sort=False)}
    for sess in sessions_order.itertuples(index=False):
        key2 = (int(sess.burst_id), int(sess.session))
        full_key = (int(uid), key2[0], key2[1])
        if full_key in test_key_set:
            alpha_before[full_key] = alpha.copy()
            seen_before[full_key] = set(seen)
            cat_ema_before[full_key] = {int(k): float(v) for k, v in ema_by_cat.items()}
        g = session_rows_lookup.get(key2)
        if g is not None:
            for row in g.sort_values(['time', 'video_id']).itertuples(index=False):
                cat = int(row.i_cat_level1_id)
                y = float(row.y_complete)
                ema_by_cat[cat] = y if cat not in ema_by_cat else EMA_ALPHA * y + (1.0 - EMA_ALPHA) * ema_by_cat[cat]
                seen.add(int(row.video_id))
        alpha += counts_lookup.get(key2, 0.0)

missing_alpha = [key for key in test_key_set if key not in alpha_before]
if missing_alpha:
    raise ValueError(f'Missing reconstructed alpha for {len(missing_alpha)} test sessions')

real_counts = struct_interactions[struct_interactions['user_id'].isin(test_users)].groupby(['user_id', 'burst_id', 'session', 'i_cat_level1_id'], observed=True).size().rename('n').reset_index()
real_count_lookup = {}
for r in real_counts.itertuples(index=False):
    key = (int(r.user_id), int(r.burst_id), int(r.session))
    real_count_lookup.setdefault(key, np.zeros(len(cat_ids), dtype=np.float64))
    real_count_lookup[key][cat_to_idx[int(r.i_cat_level1_id)]] += float(r.n)

session_state = proc_test.sort_values(['user_id', 'burst_id', 'session', 'time', 'video_id'], kind='mergesort').groupby(['user_id', 'burst_id', 'session'], observed=True, as_index=False).first()
state_lookup = {(int(r.user_id), int(r.burst_id), int(r.session)): r._asdict() for r in session_state.itertuples(index=False)}
print('alpha sessions reconstructed:', len(alpha_before))


Rebuild alpha progress:   0%|          | 0/1 [00:00<?, ?user/s]

alpha sessions reconstructed: 5


## 4. Score Candidates and Compute Treatment

For each test session, score candidate videos, take the top `n_t`, update beliefs under real and toy category counts, and compute raw, toy, and debiased treatment components.


In [5]:
user_static_cols = [c for c in continuous_cols + binary_cols + categorical_cols if c.startswith('u_')]
video_static_cols = [c for c in continuous_cols + binary_cols + categorical_cols if c.startswith('i_')]
context_history_cols = [
    'ctx_hour_sin', 'ctx_hour_cos', 'ctx_is_weekend',
    'hist_ema_y_complete', 'hist_ema_y_long', 'hist_ema_y_rewatch', 'hist_ema_y_neg', 'hist_ema_watchratio',
    'hist_cat_entropy_l2', 'hist_prev_sess_len', 'hist_intersess_gap_h',
]
user_static = proc_test.sort_values(['user_id', 'time', 'video_id'], kind='mergesort').groupby('user_id', observed=True, as_index=False).first()[['user_id'] + user_static_cols]
# Use the full processed table for the catalog, so every candidate video is available.
catalog_cols = sorted(set(['video_id'] + video_static_cols))
video_catalog_source = pd.read_parquet(PROCESSED_DATA_PATH, columns=catalog_cols + ['time'])
video_catalog = video_catalog_source.sort_values(['video_id', 'time'], kind='mergesort').groupby('video_id', observed=True, as_index=False).first()[['video_id'] + video_static_cols]

def make_candidate_frame(uid, burst_id, session, state_row):
    cand = video_catalog.copy()
    key = (int(uid), int(burst_id), int(session))
    if EXCLUDE_PREVIOUSLY_SEEN:
        seen = seen_before.get(key, set())
        if seen:
            cand = cand[~cand['video_id'].isin(seen)].copy()
    user_row = user_static[user_static['user_id'].eq(uid)]
    if user_row.empty:
        raise ValueError(f'missing user static row for user {uid}')
    user_row = user_row.iloc[0]
    for col in user_static_cols:
        cand[col] = user_row[col]
    for col in context_history_cols:
        cand[col] = state_row[col]
    cand['burst_id'] = int(burst_id)
    if 'hist_cat_ema_complete' in continuous_cols:
        cat_ema = cat_ema_before.get(key, {})
        cand['hist_cat_ema_complete'] = cand['i_cat_level1_id'].map(lambda x: cat_ema.get(int(x), np.nan))
    return cand

def moments_from_belief(B, user_idx):
    mu = mu_by_user[int(user_idx)]
    sig = sigmas_by_user[int(user_idx)]
    imp_mean = float(np.sum(B * mu))
    second = float(np.sum(B * (sig ** 2 + mu ** 2)))
    imp_std = float(np.sqrt(max(second - imp_mean ** 2, 0.0)))
    return imp_mean, imp_std

def update_counts(alpha_vec, count_vec, user_idx):
    alpha_after = alpha_vec + count_vec
    B = alpha_after / np.maximum(alpha_after.sum(), 1e-12)
    imp_mean, imp_std = moments_from_belief(B, user_idx)
    return B, imp_mean, imp_std

records = []
skipped = []
candidate_counts = []
for tr in tqdm(test_transitions.itertuples(index=False), total=len(test_transitions), desc='Candidate scoring progress', unit='session'):
    key = (int(tr.user_id), int(tr.burst_id), int(tr.session))
    try:
        state_row = state_lookup[key]
        alpha_vec = alpha_before[key]
        B_before = alpha_vec / alpha_vec.sum()
        imp_mean_before, imp_std_before = moments_from_belief(B_before, int(tr.user_idx_int))
        count_real = real_count_lookup.get(key, np.zeros(len(cat_ids), dtype=np.float64))
        n_t = int(round(float(tr.n_interactions_t)))
        if int(count_real.sum()) != n_t:
            raise ValueError(f'real count sum {count_real.sum()} != n_t {n_t}')
        cand = make_candidate_frame(int(tr.user_id), int(tr.burst_id), int(tr.session), state_row)
        if len(cand) < n_t:
            raise ValueError(f'candidate count {len(cand)} < n_t {n_t}')
        probs = predict_probs(cand)
        score = (
            float(head_weights['w_complete']) * probs[:, 0]
            + float(head_weights['w_long']) * probs[:, 1]
            + float(head_weights['w_rewatch']) * probs[:, 2]
            + float(head_weights['w_neg']) * probs[:, 3]
        )
        top_idx = np.argpartition(-score, n_t - 1)[:n_t]
        top_idx = top_idx[np.argsort(-score[top_idx])]
        top_cats = cand.iloc[top_idx]['i_cat_level1_id'].to_numpy(dtype=np.int64)
        count_toy = np.zeros(len(cat_ids), dtype=np.float64)
        for cat in top_cats:
            count_toy[cat_to_idx[int(cat)]] += 1.0
        if int(count_toy.sum()) != n_t:
            raise ValueError(f'toy count sum {count_toy.sum()} != n_t {n_t}')
        B_real_after, imp_mean_real, imp_std_real = update_counts(alpha_vec, count_real, int(tr.user_idx_int))
        B_toy_after, imp_mean_toy, imp_std_toy = update_counts(alpha_vec, count_toy, int(tr.user_idx_int))
        raw_delta_mu = imp_mean_real - imp_mean_before
        raw_delta_sigma = imp_std_real - imp_std_before
        toy_delta_mu = imp_mean_toy - imp_mean_before
        toy_delta_sigma = imp_std_toy - imp_std_before
        p_real = count_real / n_t
        p_toy = count_toy / n_t
        rec = {
            'user_id': int(tr.user_id), 'user_idx_int': int(tr.user_idx_int), 'burst_id': int(tr.burst_id),
            'session': int(tr.session), 'next_session': int(tr.next_session),
            'tau_t': float(tr.tau_t), 'tau_t1': float(tr.tau_t1), 'delta_tau': float(tr.delta_tau),
            'pred_delta_tau': float(tr.pred_delta_tau), 'delta_tau_resid': float(tr.delta_tau_resid),
            'n_t': n_t, 'candidate_count': int(len(cand)),
            'imp_mean_before': imp_mean_before, 'imp_std_before': imp_std_before,
            'imp_mean_real_after': imp_mean_real, 'imp_std_real_after': imp_std_real,
            'imp_mean_toy_after': imp_mean_toy, 'imp_std_toy_after': imp_std_toy,
            'raw_delta_mu': raw_delta_mu, 'raw_delta_sigma': raw_delta_sigma,
            'toy_delta_mu': toy_delta_mu, 'toy_delta_sigma': toy_delta_sigma,
            'D_mu': raw_delta_mu - toy_delta_mu,
            'D_sigma': raw_delta_sigma - toy_delta_sigma,
        }
        for idx, cid in enumerate(cat_ids):
            rec[f'p_real_k_{idx:02d}'] = float(p_real[idx])
            rec[f'p_toy_k_{idx:02d}'] = float(p_toy[idx])
            rec[f'count_real_k_{idx:02d}'] = float(count_real[idx])
            rec[f'count_toy_k_{idx:02d}'] = float(count_toy[idx])
        records.append(rec)
        candidate_counts.append(len(cand))
    except Exception as exc:
        skipped.append({'user_id': int(tr.user_id), 'burst_id': int(tr.burst_id), 'session': int(tr.session), 'reason': repr(exc)})

treatment = pd.DataFrame(records)
print('treatment rows:', len(treatment), 'skipped:', len(skipped))
if len(treatment):
    display(treatment[['n_t', 'candidate_count', 'D_mu', 'D_sigma', 'raw_delta_mu', 'toy_delta_mu']].describe().T)
if skipped:
    display(pd.DataFrame(skipped).head())


Candidate scoring progress:   0%|          | 0/5 [00:00<?, ?session/s]

treatment rows: 5 skipped: 0


,count,mean,std,min,25%,50%,75%,max
n_t,5.0,59.000000,20.676073,38.000000,41.000000,55.000000,79.000000,82.000000
candidate_count,5.0,10570.000000,77.562878,10460.000000,10536.000000,10572.000000,10622.000000,10660.000000
D_mu,5.0,0.062544,0.021247,0.031033,0.058720,0.062238,0.071343,0.089388
D_sigma,5.0,-0.031413,0.017588,-0.057258,-0.037406,-0.026619,-0.026401,-0.009381
raw_delta_mu,5.0,0.030011,0.025889,0.008042,0.010366,0.020101,0.041866,0.069679
toy_delta_mu,5.0,-0.032533,0.031995,-0.069287,-0.060977,-0.022991,-0.016854,0.007442


## 5. Save Treatment Outputs

The saved treatment file is transition-level and includes category counts/distributions for every level-1 category.


In [6]:
def summarize_series(s):
    if len(s) == 0:
        return {}
    return s.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict()

if len(treatment):
    treatment.to_parquet(treatment_output_path, index=False)
else:
    pd.DataFrame().to_parquet(treatment_output_path, index=False)

metrics = {
    'mode': 'full' if RUN_FULL_TREATMENT else 'smoke',
    'run_full_treatment': bool(RUN_FULL_TREATMENT),
    'num_test_transitions_all': int(len(test_full)),
    'num_test_transitions_this_run': int(len(test_transitions)),
    'num_scored_sessions': int(len(treatment)),
    'num_skipped_sessions': int(len(skipped)),
    'skipped_sessions': skipped[:100],
    'avg_candidate_count': None if not candidate_counts else float(np.mean(candidate_counts)),
    'avg_n_t': None if len(treatment) == 0 else float(treatment['n_t'].mean()),
    'D_mu_summary': summarize_series(treatment['D_mu']) if len(treatment) else {},
    'D_sigma_summary': summarize_series(treatment['D_sigma']) if len(treatment) else {},
    'raw_delta_mu_summary': summarize_series(treatment['raw_delta_mu']) if len(treatment) else {},
    'toy_delta_mu_summary': summarize_series(treatment['toy_delta_mu']) if len(treatment) else {},
    'raw_delta_sigma_summary': summarize_series(treatment['raw_delta_sigma']) if len(treatment) else {},
    'toy_delta_sigma_summary': summarize_series(treatment['toy_delta_sigma']) if len(treatment) else {},
    'D_mu_D_sigma_correlation': None if len(treatment) < 2 else float(treatment[['D_mu', 'D_sigma']].corr().iloc[0, 1]),
    'loaded_from_11c': {
        'model': str(model_path), 'preprocess': str(preprocess_path), 'features': str(features_path),
        'head_weights': str(head_weights_path), 'masked_sessions': str(masked_sessions_path),
    },
    'head_weights': head_weights,
    'exclude_previously_seen': bool(EXCLUDE_PREVIOUSLY_SEEN),
    'output_paths': {'treatment': str(treatment_output_path), 'metrics': str(metrics_output_path)},
}
metrics_output_path.write_text(json.dumps(metrics, indent=2, allow_nan=True) + '\n')

print('saved treatment:', treatment_output_path)
print('saved metrics:', metrics_output_path)


saved treatment: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_debiased_treatment_transitions_smoke.parquet
saved metrics: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_debiased_treatment_metrics_smoke.json
